# 03 - RRAO Allocation and Breakdown

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [RRAO package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: one RRAO positions table with deterministic position identifiers, legal-entity and desk lineage, gross effective notional, evidence type, product descriptors, and exclusion evidence fields. The machine-readable contract is [`frtb_rrao.positions.schema.json`](../../../docs/schemas/input_table/frtb_rrao.positions.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

RRAO is **additive**: there is no diversification, no correlation, no netting.
This means the total capital can be deterministically sliced and attributed
along any dimension — by desk, legal entity, evidence type, or position.

This notebook uses `build_rrao_allocation_reports` to show three views:
- By trading desk
- By legal entity
- By evidence type

Regulatory anchors: Basel MAR23.8(2); U.S. NPR 2.0 proposed section `__.211(c)(1)`.  
Demonstration caution: outputs are synthetic fixture evidence.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import Markdown, display

from examples.rrao_fixture import load_context, load_positions, PROFILE_US_NPR
from frtb_rrao import (
    calculate_rrao_capital,
    build_rrao_allocation_reports,
    RraoAllocationDimension,
    SUPPORTED_RRAO_ALLOCATION_DIMENSIONS,
)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ctx = load_context()
positions = load_positions()
result = calculate_rrao_capital(positions, context=ctx)

reports = build_rrao_allocation_reports(result, dimensions=SUPPORTED_RRAO_ALLOCATION_DIMENSIONS)
report_by_dim = {r.dimension: r for r in reports}

print(f"Total RRAO: ${result.total_rrao:,.0f}")
print(f"Supported allocation dimensions: {[d.value for d in SUPPORTED_RRAO_ALLOCATION_DIMENSIONS]}")

## Public API happy path

This notebook computes capital through the top-level entrypoint, then uses the public allocation report API for additive breakdowns.


In [ ]:
from frtb_rrao import calculate_rrao_capital, build_rrao_allocation_reports

public_result = calculate_rrao_capital(positions, context=ctx)
public_reports = build_rrao_allocation_reports(public_result)
print(f"Total RRAO capital: {public_result.total_rrao:,.2f}")
print(f"Allocation reports: {len(public_reports)}")
print(f"Input hash        : {public_result.input_hash}")
print(f"Profile hash      : {public_result.profile_hash}")


## Implementation details / math verification - Allocation by Trading Desk

Each bucket contains the deterministic add-on attributable to that desk.  Because
RRAO is additive, the desk allocations sum exactly to the total.  The allocation
also records gross notional (included + excluded) so a capital officer can see
the exclusion scope at the desk level.

In [ ]:
desk_report = report_by_dim[RraoAllocationDimension.DESK]
# Sort by add-on descending, push zero buckets to bottom
buckets = sorted(desk_report.buckets, key=lambda b: b.add_on, reverse=True)

desk_names = [b.bucket_key for b in buckets]
addons = [b.add_on for b in buckets]
notionals = [b.gross_effective_notional / 1e6 for b in buckets]
incl_counts = [b.line_count - b.excluded_line_count for b in buckets]
excl_counts = [b.excluded_line_count for b in buckets]

# Table
rows = []
for b in buckets:
    rows.append([
        b.bucket_key,
        f"${b.gross_effective_notional/1e6:,.0f}M",
        str(b.line_count - b.excluded_line_count),
        str(b.excluded_line_count),
        f"${b.add_on:,.0f}",
        f"{b.add_on / desk_report.total_rrao:.1%}" if desk_report.total_rrao > 0 else "0%",
    ])

display(Markdown(
    "| Desk | Gross Notional | Included | Excluded | Add-on | % of Total |\n"
    "|---|---|---|---|---|---|\n"
    + "\n".join(f"| {' | '.join(r)} |" for r in rows)
))

# Chart
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(desk_names))
bars = ax.bar(x, addons, color="#4a8fcc", alpha=0.86)
for bar, v in zip(bars, addons):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
                f"${v:,.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(desk_names, rotation=30, ha="right")
ax.set_title("RRAO capital add-on by trading desk — US NPR 2.0")
ax.set_ylabel("USD add-on")
plt.tight_layout()
plt.show()

## Allocation by Legal Entity

The legal-entity dimension allows capital officers to see the RRAO contribution
from each booking entity.  Where an entity books both included and excluded
positions the allocation preserves both the charged notional and the excluded
notional for completeness.

In [ ]:
le_report = report_by_dim[RraoAllocationDimension.LEGAL_ENTITY]
le_buckets = sorted(le_report.buckets, key=lambda b: b.add_on, reverse=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

le_names = [b.bucket_key for b in le_buckets]
le_addons = [b.add_on for b in le_buckets]
le_incl_notional = [b.gross_effective_notional / 1e6 for b in le_buckets]

# Capital by LE
bars1 = ax1.bar(le_names, le_addons, color="#7a5cc0", alpha=0.86)
for bar, v in zip(bars1, le_addons):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3000,
             f"${v:,.0f}", ha="center", va="bottom", fontsize=9)
ax1.set_title("Capital add-on by legal entity")
ax1.set_ylabel("USD add-on")

# Gross notional by LE (stacked included/excluded)
incl_n = [sum(p.gross_effective_notional for p in positions
              if p.legal_entity == b.bucket_key
              and p.position_id in b.included_position_ids) / 1e6
          for b in le_buckets]
excl_n = [sum(p.gross_effective_notional for p in positions
              if p.legal_entity == b.bucket_key
              and p.position_id in b.excluded_position_ids) / 1e6
          for b in le_buckets]

ax2.bar(le_names, incl_n, color="#7a5cc0", alpha=0.86, label="Included")
ax2.bar(le_names, excl_n, bottom=incl_n, color="#52a052", alpha=0.60, label="Excluded")
ax2.set_title("Gross notional by legal entity (USD M)")
ax2.set_ylabel("USD millions")
ax2.legend()

fig.suptitle("RRAO Allocation by Legal Entity — US NPR 2.0", fontsize=13)
plt.tight_layout()
plt.show()

for b in le_buckets:
    pct = b.add_on / le_report.total_rrao * 100 if le_report.total_rrao > 0 else 0
    print(f"{b.bucket_key}: add-on ${b.add_on:,.0f} ({pct:.1f}%)  "
          f"included {b.line_count - b.excluded_line_count} pos / "
          f"excluded {b.excluded_line_count} pos")

## Allocation by Evidence Type

Slicing by evidence type shows the regulatory risk categories driving capital.
This view is useful for model validation: the largest evidence-type buckets
should correspond to the product types where the firm has the most residual
risk exposure.

In [ ]:
ev_report = report_by_dim[RraoAllocationDimension.EVIDENCE_TYPE]
ev_buckets = sorted(
    [b for b in ev_report.buckets if b.add_on > 0],
    key=lambda b: b.add_on, reverse=True
)

ev_names = [b.bucket_key.replace("_", "\n") for b in ev_buckets]
ev_addons = [b.add_on for b in ev_buckets]

# Colour by classification: look up via result lines
ev_cls_map: dict[str, str] = {}
for line in result.lines:
    ev_cls_map[line.evidence_type.value] = line.classification.value

cls_palette = {
    "EXOTIC": "#e05252",
    "OTHER_RESIDUAL_RISK": "#e0a040",
    "SUPERVISOR_DIRECTED": "#d96e28",
    "EXCLUDED": "#52a052",
}
ev_colors = [cls_palette.get(ev_cls_map.get(b.bucket_key, "EXCLUDED"), "#aaaaaa")
             for b in ev_buckets]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ev_names))
bars = ax.bar(x, ev_addons, color=ev_colors, alpha=0.86)
for bar, v in zip(bars, ev_addons):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
            f"${v:,.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(ev_names, fontsize=8)
ax.set_title("RRAO capital add-on by evidence type — US NPR 2.0 (included positions only)")
ax.set_ylabel("USD add-on")

patches = [mpatches.Patch(color=cls_palette[c], label=c)
           for c in ["EXOTIC", "OTHER_RESIDUAL_RISK", "SUPERVISOR_DIRECTED"]]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.show()

## Allocation Reconciliation

Because RRAO is additive, each allocation dimension must sum exactly to the total.
This cell verifies reconciliation for all three dimensions.

In [ ]:
for report in reports:
    allocated = report.allocated_rrao
    total = report.total_rrao
    diff = abs(allocated - total)
    status = "OK" if diff < 1e-6 else f"FAIL (diff={diff:.2e})"
    print(f"{report.dimension.value:<20}: allocated=${allocated:,.0f}  total=${total:,.0f}  [{status}]")

## See also

- [RRAO package journey](../docs/PACKAGE_JOURNEY.md)
- [RRAO dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
